#  Análise Exploratória da base Varejo 

## 1. Script em Python para Análise Exploratória da base Varejo.


### Importando as blibliotecas necessárias.

In [ ]:
# Importando bibliotecas do Python necessárias
import pandas as pd
import numpy as np
import csv

## Sprint 1 Realização da importação dos dados.

In [ ]:
# Importando o arquivo
with open("base_varejo.csv", mode="r", encoding="utf-8") as arquivo:
    leitor = csv.DictReader(arquivo, delimiter=";")
    dados = list(leitor)  # Carrega os dados para uma lista

# Exibe o primeiro registro lido para confirmar
print(dados[0])

### Utilizando o comando pd.read_csv para carregar o aruivo.

In [ ]:
# Carregando o arquivo CSV para o DataFrame OBS: O arquivo CSV deve estar no mesmo diretório do notebook
df = pd.read_csv("base_varejo.csv", sep=";")

### Informações do Data Frame

In [ ]:
# Exibir as primeiras linhas da tabela
df.head()

In [ ]:
# Verificando características do DataFrame
df.info()


In [ ]:
#Exibindo as últimas linhas do DataFrame
df.tail()

In [ ]:
# Descrevendo estatísticas descritivas do DataFrame
df.describe()


## Sprint 2 Normalização dos dados

### A coluna Data esta o formato object. Vamos utilizar o Pandas para converter a coluna Data para o formato datetime. Assim é possível trabalhar as funcionalidades utilizando a coluna Data.

In [ ]:
# 2. Converter a coluna DATA de 'object' para 'datetime'
df['DATA'] = pd.to_datetime(df['DATA'], dayfirst=True, errors='coerce')

In [ ]:
# Verificando a conversão da coluna DATA
df.info()

In [ ]:
# Removendo espaços em branco na identificação das colunas do Data Frame
df.columns = df.columns.str.strip()

In [ ]:
# Convertendo a coluna PR_CAT para string
df['PR_CAT'] = df['PR_CAT'].astype(str)

In [ ]:
# Convertendo a coluna PR_NOME para string
df['PR_NOME'] = df['PR_NOME'].astype(str)

In [ ]:
# Definindo a primeira letra de cada palavra em maiúscula para as colunas CL_GENERO, CL_SEG, PR_CAT e PR_NOME
df['CL_GENERO'] = df['CL_GENERO'].str.title()
df['CL_SEG']    = df['CL_SEG'].str.title()
df['PR_CAT']    = df['PR_CAT'].str.title()
df['PR_NOME']   = df['PR_NOME'].str.title()

In [ ]:
# Removendo espaços em branco no início e no final das strings das colunas do Data Frame
df["CL_GENERO"] = df["CL_GENERO"].str.strip()
df["CL_SEG"]    = df["CL_SEG"].str.strip()
df["PR_CAT"]    = df["PR_CAT"].str.strip()
df["PR_NOME"]   = df["PR_NOME"].str.strip()

## Sprint 3 (Limpeza de Nulos e Duplicatas): Aplicação das condicionais e funções  para identificação e substituição de valores vazios e  de str para valores de data tipo datetime, na tabela de varejo. 

In [ ]:
# Retorna a quantidade de nulos por coluna
df.isna().sum()

In [ ]:
# Retorna o total geral de nulos no DataFrame inteiro
df.isna().sum().sum()

In [ ]:
# Conta textos vazios ou formados apenas por espaços nas colunas do tipo 'object'
df.select_dtypes(include='object').apply(lambda col: col.str.strip().eq('')).sum()

In [ ]:
# Conta textos vazios ou formados apenas por espaços nas colunas do tipo 'object'
df.select_dtypes(include='object').apply(lambda col: col.str.strip().eq('')).sum()

In [ ]:
# Verificando a quantidade de duplicatas no DataFrame
df.duplicated().sum()

In [ ]:
# O parâmetro keep=False mostra todas as ocorrências das linhas repetidas
df[df.duplicated(keep=False)]

In [ ]:
# Removendo duplicatas do DataFrame, mantendo a primeira ocorrência.
df.drop_duplicates()

## Sprint 4 Estatística Descritiva

In [ ]:
# Função para saber quem compra mais por filho.
compras_por_filhos = (
    df.groupby("CL_FHL")
    .size()
    .reset_index(name="TOTAL_COMPRAS")
    .sort_values(by="TOTAL_COMPRAS", ascending=False)
)

print(compras_por_filhos)

In [ ]:
# Quem compra mais por segmento.
compras_por_segmento = (
    df.groupby("CL_SEG")
    .size()
    .reset_index(name="TOTAL_COMPRAS")
    .sort_values(by="TOTAL_COMPRAS", ascending=False)
)

print(compras_por_segmento)

In [ ]:
# Qual categoria de produto é mais vendida.
categorias_mais_vendidas = (
    df.groupby("PR_CAT")
    .size()
    .reset_index(name="TOTAL_VENDAS")
    .sort_values(by="TOTAL_VENDAS", ascending=False)
)

print(categorias_mais_vendidas)

### Aplicando desconto promocional para a caterigora Acessorios. Para aumentar as vendas nessa categoria. Já que foi que obteve um número menor de vendas.

In [ ]:
# Aplicando desconto e sinalizando a promoção para produtos da categoria "ACESSORIOS".
def verificar_promocao(linha):
    if linha['PR_CAT'] == 'ACESSORIOS':
        return 'Sim'
    else:
        return 'Não'

def calcular_desconto_acessorios(linha):
    if linha['PR_CAT'] == 'ACESSORIOS':
        return 0.10
    else:
        return 0.0

df['EM_PROMOCAO'] = df.apply(verificar_promocao, axis=1)
df['DESCONTO_PRODUTO'] = df.apply(calcular_desconto_acessorios, axis=1)

## Análisando métricas a partir da quantidade de filhos.

In [ ]:
# Dados da coluna CL_FHL
df['CL_FHL'].describe()

In [ ]:
# Estatísticas básicas da coluna CL_FHL
print("Média:", df['CL_FHL'].mean())          # Média aritmética
print("Mediana:", df['CL_FHL'].median())       # Valor central (50%)
print("Moda:", df['CL_FHL'].mode()[0])         # Quantidade de filhos mais comum
print("Desvio Padrão:", df['CL_FHL'].std())    # Variabilidade/dispersão dos dados
print("Mínimo:", df['CL_FHL'].min())           # Menor número de filhos
print("Máximo:", df['CL_FHL'].max())           # Maior número de filhos

In [ ]:
# Tabela com contagem total e percentual de clientes por quantidade de filhos
distribuicao = pd.DataFrame({
    'Qtd_Clientes': df['CL_FHL'].value_counts().sort_index(),
    'Percentual (%)': (df['CL_FHL'].value_counts(normalize=True).sort_index() * 100).round(2)
})

print(distribuicao)

In [ ]:
# Resumo estatístico de filhos agrupado por classe/segmento
filhos_por_classe = df.groupby('CL_SEG')['CL_FHL'].agg(
    media_filhos=('mean'),
    total_filhos=('sum'),
    qtd_clientes=('count')
).reset_index()

# Arredonda a média para 2 casas decimais
filhos_por_classe['media_filhos'] = filhos_por_classe['media_filhos'].round(2)

print(filhos_por_classe)

### Aplicando desconto proporcional a quantidade de filhos. A quantidade de pessoas com 4 filhor é a menor, por isso não afetará muito na receita final, mesmo com a aplicação do desconto.

In [ ]:
# Aplicando desconto de 5%, no valor da compra total, para cada filho do cliente.

def calcular_desconto(linha):
    filhos = linha['CL_FHL']
    if filhos > 0:
        return filhos * 0.05
    else:
        return 0.0

# Criando coluna desconto
df['PERCENTUAL_DESCONTO'] = df.apply(calcular_desconto, axis=1)

# 3. Criar a nova tabela usando GROUPBY
tabela_desconto_por_filhos = df.groupby('CL_FHL').agg(
    PERCENTUAL_DESCONTO=('PERCENTUAL_DESCONTO', 'first'), 
    TOTAL_CLIENTES=('CL_FHL', 'count')                     
).reset_index()

# Exibindo a nova tabela
print(tabela_desconto_por_filhos)

## Verificando qual foi o mês com menor quantidade de itens vendidos.

In [ ]:
# Data já foi convertida para datetime, agora vamos criar a coluna MES com o nome do mês correspondente.
# 
df["MES"] = df["DATA"].dt.month_name(locale="pt_BR")
vendas_por_mes = (
    df.groupby('MES')
    .size()
    .reset_index(name='TOTAL_VENDAS')
    .sort_values(by='TOTAL_VENDAS', ascending=True)
)

print(vendas_por_mes)

## Sprint 5 Relatório e Documentação:

### # Análise Exploratória de Dados - Varejo

Este repositório contém o projeto de Análise Exploratória de Dados (EDA) e higienização da base de dados de varejo composta por 830.000 registros. O estudo abrange desde o tratamento estrutural e limpeza de duplicadas até a geração de inteligência de negócios com regras de desconto e análise sazonal.

## Sobre o Projeto

O objetivo do projeto é transformar dados brutos de transações de varejo em insights estratégicos para a tomada de decisão. A análise identifica o comportamento de compra por perfil de cliente (estado civil, classe social e quantidade de filhos) e sazonalidade das vendas, propondo ações promocionais direcionadas para categorias de baixo desempenho.

## Principais Insights

* **Sazonalidade Crítica:** Novembro registrou o menor volume de vendas do ano (40.912 transações), enquanto Janeiro atingiu o ápice com 83.963 vendas.
* **Liderança por Categoria:** A categoria de **Alimentos** lidera com 434.767 vendas, ao passo que **Acessorios** apresentou o menor desempenho (14.557 itens), justificando uma campanha promocional com 10% de desconto.
* **Concentração de Segmento:** O **Segmento B** representa a maior fatia do volume de compras (530.163 transações), seguido pelo Segmento C (232.101) e Segmento A (67.736).
* **Perfil Familiar:** 52,47% dos clientes não possuem filhos. A média geral é de 1,15 filho por cliente, mantendo-se uniforme entre as classes sociais.
* **Viabilidade Promocional por Filhos:** A concessão de desconto de 5% por filho (com teto de 20% para 4 filhos) afeta a menor parcela da base, visto que clientes com 4 filhos correspondem a apenas 9,68% do total.
* **Qualidade dos Dados:** Identificou-se a presença de 96.553 linhas duplicadas e 4 colunas nulas (`Unnamed: 10` a `Unnamed: 13`), tratadas na fase de saneamento de dados.

## Sprints do Projeto

| **Sprint**   | **Etapa**               | **Descrição das Atividades**                                                                                               |
| ------------------ | ----------------------------- | ---------------------------------------------------------------------------------------------------------------------------------- |
| **Sprint 1** | Importação dos Dados        | Leitura preliminar com`csv.DictReader`e carregamento do DataFrame via`pd.read_csv`com delimitador.                             |
| **Sprint 2** | Normalização                | Conversão da coluna`DATA`para`datetime`, remoção de espaços com`str.strip()`e padronização de maiúsculas/minúsculas. |
| **Sprint 3** | Limpeza de Nulos e Duplicatas | Mapeamento de valores nulos (`.isna()`)e remoção de 96.553 duplicatas com`df.drop_duplicates()`.                             |
| **Sprint 4** | Estatística Descritiva       | Agrupamento por segmento, categoria e número de filhos, além do cálculo das regras promocionais.                                |
| **Sprint 5** | Relatório & Documentação   | Consolidação dos indicadores finais de desempenho e elaboração da documentação técnica.                                     |
| **Sprint 6** | Versionamento                 | Publicação do código-fonte, dados limpos e documentação no repositório GitHub.                                               |

## Pré-requisitos e Configuração do VS Code

Para reproduzir ou contribuir com o projeto, siga o passo a passo para instalação e configuração do ambiente de desenvolvimento no  **Visual Studio Code** .

### 1. Instalação das Ferramentas Básicas

* **Python 3.11+** : Faça o download e instale a versão oficial do [Python](https://www.python.org/downloads/). Marque a opção **"Add Python to PATH"** durante a instalação.
* **VS Code** : Instale o editor de código oficial pelo site do [Visual Studio Code](https://code.visualstudio.com/).

### 2. Extensões Recomendadas no VS Code

Abra o VS Code, acesse o menu de extensões (`Ctrl + Shift + X` ou `Cmd + Shift + X` no macOS) e instale as seguintes extensões:

* **Python**  *(Microsoft)* : Adiciona suporte completo à linguagem Python, com suporte a autocompletar (IntelliSense) e depuração.
* **Jupyter**  *(Microsoft)* : Permite executar, editar e visualizar arquivos de Jupyter Notebook (`.ipynb`) diretamente no VS Code.
* **Data Wrangler**  *(Microsoft)* : Interface gráfica para exploração e limpeza rápida de dados integrada aos DataFrames do Pandas.
* **Office Viewer / Excel Viewer** : Permite visualizar e inspecionar planilhas `.csv` e `.xlsx` sem precisar sair do editor.

## Tecnologias e Bibliotecas Utilizadas

* **Python 3.11;**
* **Pandas** (Tratamento, agregação e manipulação de dados);
* **NumPy** (Apoio a cálculos estruturados);
* **Módulo CSV** (Leitura estruturada via dicionários);

## Como Executar

1. Clone o repositório para o seu ambiente local.
2. Posicione o arquivo `base_varejo.csv` no mesmo diretório do notebook `Mini_projeto.ipynb`.
3. Abra o VS Code na pasta do projeto e selecione o kernel Python configurado.
4. Execute as células do Jupyter Notebook sequencialmente da **Sprint 1** à **Sprint 6**.


## 📝 Licença

Este projeto está público.

## Sprint 6 Versionamento: Envio dos arquivos (script + README.md + df_limpo), via Git para o repositório no GitHub.

In [ ]:
# Exporta o DataFrame limpo/alterado para um novo arquivo CSV
df.to_csv("base_varejo_limpo.csv", sep=";", index=False, encoding="utf-8-sig")

print("Arquivo exportado com sucesso!")

### O Script do projeto, juntamente com README.md e o df_limpo estão disponíveis no GitHub.  Link: https://github.com/Fernandohb55/Mini_projeto_avaliativo_modulo_1